In [16]:
import torch
import torch.nn as nn
from torch.nn import functional as F
import random
import pickle
import math

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(device)
if device == 'cpu':
    torch.set_num_threads(7)

# Hyperparameters
block_size = 128
batch_size = 64
max_iters = 4000
learning_rate = 1e-3
eval_iters = 500
n_embd = 256
n_layer = 6
n_head = 8
dropout = 0.2

cpu


In [17]:
with open('pride_and_prejudice.txt', 'r', encoding = 'utf-8') as f:
    text = f.read()
chars = sorted(set(text))
print(chars)
vocab_size = len(chars)

['\t', '\n', ' ', '!', '&', '(', ')', '*', ',', '-', '.', '/', '0', '1', '2', '3', '4', '5', '6', '7', '8', '9', ':', ';', '?', 'A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O', 'P', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y', 'Z', '[', ']', '^', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z', '{', '}', '·', 'à', 'â', 'é', 'ê', 'œ', '‘', '’', '“', '”']


In [18]:
string_to_int = { ch:i for i,ch in enumerate(chars) }
int_to_string = { i:ch for i,ch in enumerate(chars) }
encode = lambda s: [string_to_int[c] for c in s]
decode = lambda l: ''.join([int_to_string[i] for i in l])

data = torch.tensor(encode(text), dtype = torch.long)

In [19]:
n = int(0.8*len(data))
train_data = data[:n]
test_data = data[n:]

def get_batch(split):
    data_split = train_data if split == 'train' else test_data
    ix = torch.randint(len(data_split) - block_size, (batch_size,), device = device)
    # print(ix)
    offsets = torch.arange(block_size, device = device)
    x = data_split[ix.unsqueeze(1) + offsets]
    y = data_split[ix.unsqueeze(1) + offsets + 1]
    return x, y

In [20]:
@torch.no_grad()
def estimate_loss():
    out = {}
    model.eval()
    for split in ['train', 'test']:
        losses = torch.zeros(eval_iters, device = device)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            _, loss = model(X, Y)
            losses[k] = loss
        out[split] = losses.mean().item()
    model.train()
    return out

In [21]:
class MultiHeadAttention(nn.Module):
    # mutliple heads of self-attention in parallel

    def __init__(self, n_embd, n_head):
        super().__init__()
        self.n_head = n_head
        self.head_size = n_embd // n_head
        # combine K, Q, V projection in a single matrix multiplication
        self.c_attn = nn.Linear(n_embd, 3 * n_embd, bias=False)
        self.proj = nn.Linear(n_embd, n_embd)
        self.dropout_p = dropout

    def forward(self, x):
        B, T, C = x.shape
        # calculate K, Q, V for all heads in batch
        k, q, v = self.c_attn(x).chunk(3, dim=-1)

        # reshape to (B, n_head, T, head_size)
        k = k.view(B, T, self.n_head, self.head_size).transpose(1, 2)
        q = q.view(B, T, self.n_head, self.head_size).transpose(1, 2)
        v = v.view(B, T, self.n_head, self.head_size).transpose(1, 2)

        # fast fused FlashAttention kernel (handles masking and dropout internally)
        out = F.scaled_dot_product_attention(
            q, k, v, 
            is_causal=True, 
            dropout_p=self.dropout_p if self.training else 0.0
        )
        
        # assemble all head outputs side-by-side
        out = out.transpose(1, 2).contiguous().view(B, T, C)
        return self.proj(out)
        
class FeedForward(nn.Module):
    # a simple linear layer followed by a non-linearity
    
    def __init__(self, n_embd):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.GELU(),
            nn.Linear(4 * n_embd, n_embd),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)
        
class Block(nn.Module):
    # transformer block: communication followed by computation
    
    def __init__(self, n_embd, n_head):
        # n_embd: embedding dimension, n_head: the number of heads we want
        
        super().__init__()
        self.sa = MultiHeadAttention(n_embd, n_head)
        self.ffwd = FeedForward(n_embd)
        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)

    def forward(self, x):
        x = x + self.sa(self.ln1(x))
        x = x + self.ffwd(self.ln2(x))
        return x
        
class GPTLanguageModel(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        self.position_embedding_table = nn.Embedding(block_size, n_embd)
        self.emb_dropout = nn.Dropout(dropout)
        self.blocks = nn.Sequential(*[Block(n_embd, n_head = n_head) for _ in range(n_layer)]) # creare decoding layers

        self.ln_f = nn.LayerNorm(n_embd) # final layer norm
        self.lm_head = nn.Linear(n_embd, vocab_size)

        self.apply(self._init_weights)

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            torch.nn.init.normal_(module.weight, mean = 0.0, std = 0.02)
            if module.bias is not None:
                torch.nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            torch.nn.init.normal_(module.weight, mean = 0.0, std = 0.02)

    def forward(self, index, targets=None):
        B, T = index.shape

        tok_emb = self.token_embedding_table(index) # (B, T, C)
        pos_emb = self.position_embedding_table(torch.arange(T, device = device)) # (T, C)
        x = tok_emb + pos_emb # (B, T, C)
        x = self.emb_dropout(x)
        x = self.blocks(x) # (B, T, C)
        x = self.ln_f(x) # (B, T, C)
        logits = self.lm_head(x) # (B, T, vocab_size)

        if targets ==  None:
            loss = None
        else:
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1))
            
        return logits, loss

    @torch.no_grad()
    def generate(self, index, max_new_tokens):
        # index = (B, T) array of indices in the current context
        for _ in range(max_new_tokens):
            # crop context to last block_size tokens
            index_cond = index[:, -block_size:]
            # get the predictions
            logits, _ = self.forward(index_cond)
            # focus only on the last time step
            logits = logits[:, -1, :] # (B, C)
            # apply softmax to get probabilities, focusing on last dimension
            probs = F.softmax(logits, dim = -1) # (B, C)
            # sample from the distribution
            index_next = torch.multinomial(probs, num_samples = 1) # (B, 1)
            # append sampled index to the running sequence
            index = torch.cat((index, index_next), dim = 1) # (B, T+1)
        return index

# push params to GPU for more efficient training
model = GPTLanguageModel(vocab_size)
m = model.to(device)

# context = torch.zeros((1,1), dtype = torch.long, device = device)
# generated_chars = decode(m.generate(context, max_new_tokens=500)[0].tolist())
# print(generated_chars)


In [22]:
# PyTorch optimizer
optimizer = torch.optim.AdamW(model.parameters(), lr = learning_rate, weight_decay = 1e-2)
# Cosine annealing learning rate scheduler
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=max_iters, eta_min = 1e-5)

best_test_loss = float('inf')

# training loop
for iter in range(max_iters):
    if iter % eval_iters == 0:
        losses = estimate_loss()
        print(f'step: {iter}, train loss: {losses["train"]:.3f}, test loss: {losses["test"]:.3f}')

        # checkpoint the best model weights
        if losses['test'] < best_test_loss:
            best_test_loss = losses["test"]
            torch.save(model.state_dict(), 'best_model.pt')
            print(f'  --> Saved new best model checkpoint (test_loss: {best_test_loss:.3f})')
    
    # sample a batch of data
    xb, yb = get_batch('train')

    # evaluate the loss
    logits, loss = model.forward(xb, yb)
    optimizer.zero_grad(set_to_none=True) # optimize based on current data
    loss.backward()
    optimizer.step()
    scheduler.step()
print(loss.item())

step: 0, train loss: 4.622, test loss: 4.621
  --> Saved new best model checkpoint (test_loss: 4.621)
step: 500, train loss: 2.068, test loss: 2.070
  --> Saved new best model checkpoint (test_loss: 2.070)
step: 1000, train loss: 1.272, test loss: 1.315
  --> Saved new best model checkpoint (test_loss: 1.315)
step: 1500, train loss: 1.100, test loss: 1.187
  --> Saved new best model checkpoint (test_loss: 1.187)
step: 2000, train loss: 1.009, test loss: 1.150
  --> Saved new best model checkpoint (test_loss: 1.150)
step: 2500, train loss: 0.934, test loss: 1.131
  --> Saved new best model checkpoint (test_loss: 1.131)
step: 3000, train loss: 0.871, test loss: 1.128
  --> Saved new best model checkpoint (test_loss: 1.128)
step: 3500, train loss: 0.838, test loss: 1.125
  --> Saved new best model checkpoint (test_loss: 1.125)
0.906190037727356


In [33]:
m.load_state_dict(torch.load('best_model.pt', map_location=device))
m.eval()

prompt = 'Hello! Can you see me?'
context = torch.tensor(encode(prompt), dtype=torch.long, device=device)
generated_chars = decode(m.generate(context.unsqueeze(0), max_new_tokens=100)[0].tolist())
print(generated_chars)

Hello! Can you see me? I should
      wish to believe myself and mistaken, in deprive of your being
engaged in that match,
